<a href="https://colab.research.google.com/github/umar834-web/Mental-Health-RAG-Chatbot/blob/main/Mental_Health_RAG_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q transformers sentence-transformers
!pip install -q faiss-cpu chromadb
!pip install -q accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/mental-health-rag"

folders = ["data", "models", "notebooks", "outputs"]

for f in folders:
    os.makedirs(os.path.join(base_path, f), exist_ok=True)

print("✅ Folder structure created")

In [ ]:
import langchain
import faiss
import chromadb
from sentence_transformers import SentenceTransformer

print("✅ All libraries working fine")

In [ ]:
!pip install chromadb --upgrade

**DATASET**

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Amod/mental_health_counseling_conversations")

print(dataset)

In [ ]:
dataset['train'][0]

In [ ]:
data = dataset['train']

documents = []

for item in data:
    text = f"User: {item['Context']}\nTherapist: {item['Response']}"
    documents.append(text)

print("Total documents:", len(documents))

In [ ]:
import os

save_path = "/content/drive/MyDrive/mental-health-rag/data/mental_health_docs.txt"

with open(save_path, "w") as f:
    for doc in documents:
        f.write(doc + "\n\n")

print("✅ Dataset saved successfully")

**CHUNKING**

In [ ]:
from langchain_community.document_loaders import TextLoader

file_path = "/content/drive/MyDrive/mental-health-rag/data/mental_health_docs.txt"

loader = TextLoader(file_path)
documents = loader.load()

print("Total documents:", len(documents))

In [ ]:
print(documents[0].page_content[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Config A (small chunks)
splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Config B (large chunks)
splitter_1000 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks_500 = splitter_500.split_documents(documents)
chunks_1000 = splitter_1000.split_documents(documents)

print("Chunks (500):", len(chunks_500))
print("Chunks (1000):", len(chunks_1000))

In [ ]:
print(chunks_500[0].page_content[:300])

**EMBEDDINGS**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
# Model A (fast)
emb_model_1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Model B (better quality)
emb_model_2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
text = "I feel anxious and stressed"

vector = emb_model_1.embed_query(text)

print("Vector length:", len(vector))
print(vector[:5])  # first 5 values

In [ ]:
# For chunk set A
texts_500 = [doc.page_content for doc in chunks_500]
embeddings_500 = emb_model_1.embed_documents(texts_500)

print("Embeddings created:", len(embeddings_500))

In [ ]:
texts_1000 = [doc.page_content for doc in chunks_1000]
embeddings_1000 = emb_model_2.embed_documents(texts_1000)

print("Embeddings created:", len(embeddings_1000))

**VECTOR DATABASE**

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

In [ ]:
# Using small chunks + embedding model 1
db_faiss = FAISS.from_documents(
    chunks_500,
    emb_model_1
)

print("✅ FAISS DB created")

In [ ]:
db_chroma = Chroma.from_documents(
    chunks_500,
    emb_model_1,
    persist_directory="/content/drive/MyDrive/mental-health-rag/chroma_db"
)

print("✅ Chroma DB created")

In [ ]:
query = "I feel stressed and anxious"

docs_faiss = db_faiss.similarity_search(query, k=3)

for i, doc in enumerate(docs_faiss):
    print(f"\nResult {i+1}:\n", doc.page_content[:200])

In [ ]:
docs_chroma = db_chroma.similarity_search(query, k=3)

for i, doc in enumerate(docs_chroma):
    print(f"\nResult {i+1}:\n", doc.page_content[:200])

In [ ]:
print("FAISS Results:", len(docs_faiss))
print("Chroma Results:", len(docs_chroma))

In [ ]:
query = "I feel very anxious and can't sleep"

print("FAISS Response:\n")
print(rag_chat(query, retriever_faiss))

print("\n\nChroma Response:\n")
print(rag_chat(query, retriever_chroma))

**PROMPT + LLM (CORE OF RAG)**

In [ ]:
retriever_faiss = db_faiss.as_retriever(search_kwargs={"k": 3})
retriever_chroma = db_chroma.as_retriever(search_kwargs={"k": 3})

In [ ]:
def create_prompt(context, query):
    return f"""
You are a compassionate mental health assistant.

Guidelines:
- Respond in a calm, supportive, and human-like way
- Do NOT repeat the question
- Do NOT repeat sentences
- Keep answer concise (3–5 sentences)
- If user expresses distress, respond with empathy

Context:
{context}

User: {query}

Helpful Response:
"""

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=200
)

In [ ]:
print(llm("I feel anxious and stressed")[0]['generated_text'])

In [ ]:
model="tiiuae/falcon-rw-1b"

In [ ]:
def rag_chat(query, retriever):
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    return response

In [ ]:
def rag_chat(query, retriever):

    docs = retriever.invoke(query)   # ✅ FIXED

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    return response

In [ ]:
def safety_check(query):
    danger_words = ["suicide", "kill myself", "self-harm"]

    for word in danger_words:
        if word in query.lower():
            return True
    return False

In [ ]:
def rag_chat_safe(query, retriever):

    if safety_check(query):
        return """I'm really sorry you're feeling this way.
You are not alone. Please reach out to someone you trust.

📞 India Helpline: 9152987821 (AASRA)
"""

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    # Remove prompt from output
    response = response.replace(prompt, "").strip()

    return response

**EVALUATION + UI + FINALIZATION**

In [ ]:
!pip install nltk rouge-score

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

reference = ["I'm sorry you're feeling anxious. Try some relaxation techniques."]
candidate = rag_chat("I feel anxious", retriever_faiss)

score = sentence_bleu([reference[0].split()], candidate.split())

print("BLEU Score:", score)

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

scores = scorer.score(reference[0], candidate)

print(scores)

In [ ]:
def rag_chat_safe(query, retriever):

    # Safety check
    if safety_check(query):
        return "I'm really sorry you're feeling this way. Please consider reaching out to a trusted person or a helpline."

    # ✅ FIXED retrieval
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    # Optional cleanup
    response = response[len(prompt):]

    return response

In [ ]:
queries = [
    "I feel depressed",
    "I have anxiety",
    "I want to harm myself"
]

for q in queries:
    print("\nQuery:", q)
    print(rag_chat_safe(q, retriever_faiss))

In [ ]:
model="tiiuae/falcon-rw-1b"

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2
)

In [ ]:
llm = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    max_new_tokens=150,
    pad_token_id=50256
)

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py
import streamlit as st

st.title("🧠 Mental Health Chatbot")

query = st.text_input("How are you feeling today?")

if query:
    response = rag_chat_safe(query, retriever_faiss)
    st.write(response)